# Tokenization

This part will be deticated to simple tokenization for my own LLM to extend knowledge in how they work and get finally hired!

In [4]:
import requests
import json
import time

In [ ]:
def fetch_hackernews_data(num_stories=1000):
    url = "https://hacker-news.firebaseio.com/v0/topstories.json"
    story_ids = requests.get(url).json()[:num_stories]

    stories = []
    for story_id in story_ids:
        story_url = f"https://hacker-news.firebaseio.com/v0/item/{story_id}.json"
        response = requests.get(story_url).json()
        if "text" in response:
            stories.append(response["text"])
        time.sleep(0.5)

    return stories

hackernews_texts = fetch_hackernews_data()

# Save the dataset of hacker news

Not to parse the API constantly I save everything in .json file

In [ ]:
save_path = "hackernews_texts.json"
with open(save_path, "w") as f:
    json.dump(hackernews_texts, f)

# Actual tokenization

For now character-level tokenizer

In [5]:
import json

save_path = "hackernews_texts.json"
with open(save_path, "r") as f:
    hackernews_texts = json.load(f)

In [6]:
import re

def clean_text(text):
    text = re.sub(r"<[^>]+>", "", text)  # Remove HTML tags
    text = re.sub(r"[^a-zA-Z0-9 .,!?]", "", text)  # Retain only valid characters
    return text

cleaned_texts = [clean_text(text) for text in hackernews_texts]
concatenated_text = "\n".join(cleaned_texts)

char_list = list()
for letter in concatenated_text:
    char_list.append(letter)

char_list[:10]
vocabulary = sorted(list(set(char_list)))

# Convert text into numbers

In [7]:
import torch

In [21]:
stoi = {ch: number for number, ch in enumerate(vocabulary)}
encoded_text = [stoi[ch] for ch in concatenated_text]

tensored_data = torch.tensor(encoded_text, dtype=torch.long)
print(len(tensored_data))

n = int(0.9 * len(tensored_data))
train_data = tensored_data[:n]
val_data = tensored_data[n:]

len(vocabulary)

61695


68

In [10]:
def get_batch(split, block_size, batch_size):
    source = train_data if split == "train" else val_data
    
    ix = torch.randint(len(source) - block_size - 1, (batch_size,))
    
    x = torch.stack([source[i:i+block_size] for i in ix])
    y = torch.stack([source[i+1:i+block_size+1] for i in ix])
    
    return x, y

# Create embeddings for each letter with position

In [ ]:
from torch import nn

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model, block_size):
        super().__init__()

        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(block_size, d_model)

    def forward(self, x):  
        B, T = x.shape  

        token_emb = self.token_embedding(x)
        pos_ids = torch.arange(T, device=x.device)
        pos_emb = self.position_embedding(pos_ids)

        x = token_emb + pos_emb  

        return x

In [22]:
d_model = 128
vocabulary_size = len(vocabulary)
block_size = 64
batch_size = 32

model = MiniGPT(vocab_size=vocabulary_size, d_model=128, block_size=64)
xb, yb = get_batch("train", block_size, batch_size)

out = model(xb)
print(out.shape)

Shape: torch.Size([32, 64])
x min: 0
x max: 66
vocab_size: 68
token_emb: tensor([[[ 0.2548,  0.5319, -0.8833,  ...,  0.2938,  2.2451, -0.8959],
         [-1.1504, -1.5754,  0.1200,  ..., -0.3344,  1.6109,  0.5146],
         [ 0.5632,  1.1274,  0.2957,  ...,  0.0241,  1.9061, -0.0720],
         ...,
         [ 0.7791, -0.5173,  0.6959,  ...,  0.0066, -0.9078, -2.0385],
         [-2.1173, -0.4671,  0.2874,  ...,  0.5035, -1.7928, -1.8694],
         [ 1.1900,  0.0229, -2.0191,  ..., -1.0862, -0.2309, -0.3665]],

        [[-1.1504, -1.5754,  0.1200,  ..., -0.3344,  1.6109,  0.5146],
         [ 0.7791, -0.5173,  0.6959,  ...,  0.0066, -0.9078, -2.0385],
         [-0.5326,  0.6868, -0.2471,  ...,  0.8380,  0.5612, -1.5146],
         ...,
         [ 0.8727,  0.0692,  0.4566,  ..., -0.5318, -1.5481,  0.1540],
         [ 0.4179, -1.0096, -0.2023,  ..., -1.5539, -0.1935,  0.3258],
         [ 0.2548,  0.5319, -0.8833,  ...,  0.2938,  2.2451, -0.8959]],

        [[ 0.7791, -0.5173,  0.6959,  ..., 

# Model structure

In [23]:
import torch
from torch import nn
import torch.nn.functional as F

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, block_size):
        super().__init__()

        self.key = nn.Linear(d_model, d_model)
        self.query = nn.Linear(d_model, d_model)
        self.value = nn.Linear(d_model, d_model)

        self.register_buffer(
            "mask",
            torch.tril(torch.ones(block_size, block_size))
        )

    def forward(self, x):
        B, T, C = x.shape

        K = self.key(x)
        Q = self.query(x)
        V = self.value(x)

        scores = Q @ K.transpose(-2, -1) / (C ** 0.5)

        scores = scores.masked_fill(
            self.mask[:T, :T] == 0,
            float('-inf')
        )

        weights = F.softmax(scores, dim=-1)

        out = weights @ V

        return out

In [24]:
class FeedForward(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.ReLU(),
            nn.Linear(4 * d_model, d_model),
        )

    def forward(self, x):
        return self.net(x)

In [25]:
class Block(nn.Module):
    def __init__(self, d_model, block_size):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, block_size)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

In [26]:
class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model, block_size, n_layers):
        super().__init__()

        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(block_size, d_model)

        self.blocks = nn.Sequential(
            *[Block(d_model, block_size) for _ in range(n_layers)]
        )

        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

        self.block_size = block_size

    def forward(self, x, targets=None):
        B, T = x.shape

        token_emb = self.token_embedding(x)
        pos_ids = torch.arange(T, device=x.device)
        pos_emb = self.position_embedding(pos_ids)

        x = token_emb + pos_emb

        x = self.blocks(x)

        x = self.ln_f(x)

        logits = self.head(x)  # (B, T, vocab_size)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

# Training Loop

In [27]:
model = MiniGPT(vocabulary_size, d_model=128, block_size=64, n_layers=4)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

for step in range(5000):
    xb, yb = get_batch("train", block_size=64, batch_size=32)

    logits, loss = model(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 500 == 0:
        print("step:", step, "loss:", loss.item())

step: 0 loss: 4.3700642585754395
step: 500 loss: 2.362974166870117
step: 1000 loss: 1.9763896465301514
step: 1500 loss: 1.7162067890167236
step: 2000 loss: 1.4704078435897827
step: 2500 loss: 1.3598554134368896
step: 3000 loss: 1.1765341758728027
step: 3500 loss: 1.0329257249832153
step: 4000 loss: 0.9244229793548584
step: 4500 loss: 0.7710949182510376


# Usage

In [28]:
def generate(model, idx, max_new_tokens):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -model.block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, next_token), dim=1)
    return idx

In [46]:
itos = {i: ch for ch, i in stoi.items()}
encoded_text = torch.tensor([stoi[ch] for ch in "What is new in the hacker world"], dtype=torch.long)
context = encoded_text.unsqueeze(0)
#context = torch.zeros((1,1), dtype=torch.long)

context = torch.zeros((1,1), dtype=torch.long)
generated = generate(model, context, 100)

"".join([itos[int(num)] for num in generated[0]])

'\ny via the WebKit layer rout by greated. Figma. Not key.Itx27s gmaths all as users a developers and a'